# เปรียบเทียบโมเดล Object Detection

Notebook นี้ค้นหา `test_predictions.json` ของทุกโมเดล แล้วประเมินด้วย test set และกติกาเดียวกัน ผลลัพธ์แยกรายโมเดลใน `results/` พร้อม dashboard HTML

> ก่อนรัน: แก้ path ใน `config.json` เพียงไฟล์เดียว ไม่ต้องแก้โค้ดในหลาย cell

In [ ]:
# รัน cell นี้ครั้งเดียวถ้า environment ยังไม่มี dependencies
# %pip install -r requirements-local.txt

from pathlib import Path
import json, sys
from IPython.display import Image, display, HTML

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'evaluator.py').is_file() else Path.cwd() / 'model_comparison'
if not (NOTEBOOK_DIR / 'evaluator.py').is_file():
    raise FileNotFoundError('ไม่พบ evaluator.py: ให้เปิด notebook จากโฟลเดอร์โปรเจกต์หรือ model_comparison')
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))
from evaluator import discover_models, evaluate_all, display_columns

CONFIG_PATH = (NOTEBOOK_DIR / 'config.json').resolve()
print(f'Config: {CONFIG_PATH}')

## 1. ตรวจการตั้งค่าและไฟล์ที่ค้นพบ

ถ้า cell นี้แจ้งว่าไม่พบ ground truth ให้แก้ `ground_truth` ใน `config.json` ให้ชี้ไปยัง COCO test JSON ของเซิร์ฟเวอร์

In [ ]:
config, ground_truth, models = discover_models(CONFIG_PATH)
print(f'Ground truth: {ground_truth}')
print(f'พบ {len(models)} โมเดล:')
for index, model in enumerate(models, 1):
    weight = model['weight_path'] or 'ไม่พบ/ไม่ได้ระบุ'
    print(f"  {index}. {model['name']}")
    print(f"     prediction: {model['prediction_path']}")
    print(f"     weight:     {weight}")
print('\nค่าประเมิน:', {key: config[key] for key in [
    'confidence_threshold', 'iou_threshold', 'max_detections_per_image'
]})

## 2. ประเมินทุกโมเดล

Metric หลักคือ Precision, Recall, Micro/Macro F1, mAP และ AR โดยผลทั้งหมดถูกบันทึกเป็น CSV, JSON, PNG และ HTML

In [ ]:
summary = evaluate_all(CONFIG_PATH)
display(display_columns(summary))

## 3. กราฟสรุป

In [ ]:
results_dir = (CONFIG_PATH.parent / config.get('output_dir', 'results')).resolve()
display(Image(filename=str(results_dir / 'model_comparison.png')))
display(Image(filename=str(results_dir / 'best_f1_by_threshold.png')))
print(f'หน้าเว็บ: {results_dir / "dashboard.html"}')
display(HTML(f'<a href="{(results_dir / "dashboard.html").as_uri()}" target="_blank">เปิด dashboard</a>'))

## 4. เปิดหน้าเว็บ

รัน cell ด้านล่างหนึ่งครั้ง แล้วเปิดลิงก์ หาก notebook อยู่บนเซิร์ฟเวอร์ให้เปิด/forward port 8501

In [ ]:
import socket, subprocess, time

WEB_PORT = 8501
def port_is_open(port):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=0.5):
            return True
    except OSError:
        return False

if not port_is_open(WEB_PORT):
    subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run', str(NOTEBOOK_DIR / 'app.py'),
        '--server.address=0.0.0.0', f'--server.port={WEB_PORT}', '--server.headless=true'
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        if port_is_open(WEB_PORT):
            break
        time.sleep(0.5)
print(f'เปิดเว็บ: http://localhost:{WEB_PORT}')

## อ่านผลอย่างสั้น

- ใช้ **mAP@0.50:0.95** เป็นคะแนนมาตรฐานหลักสำหรับคุณภาพ detector
- ใช้ **F1** เมื่ออยากได้สมดุลระหว่าง false alarm (Precision) และการตรวจเจอ (Recall)
- ใช้ **best confidence** เป็นแนวทางตั้ง threshold ตอน deploy แต่ควรยืนยันกับ validation set ก่อน
- ดู `results/<ชื่อโมเดล>/per_class_metrics.csv` เพื่อหาคลาสที่แต่ละโมเดลยังอ่อน